# RL Room-Player — listening to an agent learn to play the room

An RL agent controls a simple synth (pitch + filter cutoff). Every sound it makes is sent
**through a room reverb**, then compared to a **goal sound** via their spectrograms.
The agent's reward is *how close it got*. You **hear every attempt, step by step**, and watch
it move from wrong to right.

Two tools, one idea: this is the *search / action-space* side of the seminar — a machine
**solving** a space — as opposed to AFTER, where **you navigate** a space by ear.

**On Kaggle:** add your audio via **Add Data** (mounts read-only at `/kaggle/input/...`) or drop files into `/kaggle/working/`. If you'll use the CLAP upgrade or pip installs, toggle **Internet → on** in the right-hand settings panel. Saved outputs go to `/kaggle/working/`.

**How to use:** Run the cells top to bottom. Start with the built-in synthetic reverb and a
synth-defined goal so it works immediately, then swap in your own room impulse response and
your own goal sound where marked.


In [ ]:
# --- setup ---
import os
import numpy as np
from scipy import signal as sps
import matplotlib.pyplot as plt
from IPython.display import Audio, display
import time

SR = 22050  # sample rate (low-ish = fast, fine for a demo)
np.random.seed(0)

OUTPUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."   # Kaggle vs. Binder/local/elsewhere

def normalize(x):
    return x / (np.max(np.abs(x)) + 1e-9)

def band_limited_saw(freq, t, max_harmonics=20):
    """Additive-synthesis sawtooth (Fourier series), harmonics capped below Nyquist -> no aliasing."""
    n = max(1, min(max_harmonics, int((SR/2) / max(freq, 1))))
    saw = np.zeros_like(t)
    for k in range(1, n+1):
        saw += (1/k) * np.sin(2*np.pi*freq*k*t)
    return saw * (2/np.pi)  # normalize toward [-1, 1]

def env_ar(n, attack_frac=0.05, release_curve=4.0):
    """Attack + exponential release envelope (replaces a symmetric clicky 'tent')."""
    n_a = max(1, int(n*attack_frac))
    attack = np.linspace(0, 1, n_a)
    release = np.exp(-release_curve * np.linspace(0, 1, n - n_a))
    return np.concatenate([attack, release]).astype(np.float32)

## 0. Your source sound — what the agent actually shapes

Instead of synthesizing a tone from scratch (numpy oscillators only sound so good), the agent
now takes a **real recorded sound** — a vocal, a field recording, anything — and learns to
shape it with real audio effects (pitch shift, distortion) to chase the goal. The "search space"
is now "how do I process this sound," not "how do I build a sound from nothing."

**To use your own:** put a short WAV (a few seconds, e.g. you singing/talking) at the path below
and set `USE_MY_SOURCE = True`. Without one, a simple built-in tone is used as a placeholder so
the notebook still runs.

In [ ]:
USE_MY_SOURCE = False  # <-- set True after you add a source recording (your voice, etc.)

# Kaggle: put source.wav in /kaggle/working/ or add it as a Dataset and edit the path
SOURCE_PATH = "/kaggle/working/source.wav"

def _fallback_tone(dur=0.5):
    """Used only if no source file is provided, so the notebook still runs out of the box."""
    t = np.linspace(0, dur, int(SR*dur), endpoint=False)
    x = band_limited_saw(220, t)
    return (x * env_ar(len(x))).astype(np.float32)

if USE_MY_SOURCE:
    from scipy.io import wavfile
    sr_s, s = wavfile.read(SOURCE_PATH)
    s = s.astype(np.float32)
    if s.ndim > 1: s = s.mean(axis=1)              # mono
    if sr_s != SR: s = sps.resample(s, int(len(s)*SR/sr_s))
    SOURCE_AUDIO = normalize(s[:int(SR*0.5)])       # trim to 0.5s, matching the synth note length
    SOURCE_LABEL = "your uploaded source"
else:
    SOURCE_AUDIO = normalize(_fallback_tone())
    SOURCE_LABEL = "built-in fallback tone (upload your own voice/sound to replace this)"

print("SOURCE:", SOURCE_LABEL)
display(Audio(SOURCE_AUDIO, rate=SR))

## 1b. Optional: a real Tone.js synth instead of hand-rolled numpy DSP

The numpy/scipy oscillator above is still not great-sounding even band-limited — getting good
synth timbre out of raw signal processing is a deep rabbit hole. This bridges the RL loop to
**Tone.js** instead: a real synthesis library with properly anti-aliased oscillators (the native
Web Audio API oscillator types are band-limited already, no manual tricks needed).

The reward loop still needs the actual samples back in numpy, so Python still drives the search
— Tone.js just renders the waveform in the browser (via `Tone.Offline`, no realtime playback
needed) and the samples come back over the standard Jupyter widget comm channel.

**This only works in a real running Jupyter kernel with a live browser attached — e.g. Binder or
local Jupyter — not Kaggle's hosted notebook environment.** If you're on Kaggle, leave
`SOUND_ENGINE = "python"` below and ignore this. `pip install anywidget` first if it's missing.

In [ ]:
import asyncio

try:
    import anywidget
    import traitlets

    class ToneBridge(anywidget.AnyWidget):
        params      = traitlets.List([0.0, 0.0]).tag(sync=True)   # [freq, cutoff]
        duration    = traitlets.Float(0.5).tag(sync=True)
        request_id  = traitlets.Int(0).tag(sync=True)              # bump to trigger a render
        response_id = traitlets.Int(-1).tag(sync=True)             # JS echoes request_id back when done
        samples     = traitlets.List([]).tag(sync=True)            # rendered waveform
        error       = traitlets.Unicode("").tag(sync=True)         # set by JS if rendering throws

        _esm = """
        function render({ model, el }) {
          async function doRender() {
            const thisId = model.get("request_id");
            try {
              const Tone = await import("https://esm.sh/tone@15");
              const [freq, cutoff] = model.get("params");
              const duration = model.get("duration");
              const buffer = await Tone.Offline(() => {
                const filter = new Tone.Filter(cutoff, "lowpass").toDestination();
                const osc = new Tone.Oscillator(freq, "sawtooth").connect(filter);
                osc.start(0);
                osc.stop(duration);
              }, duration);
              model.set("error", "");
              model.set("samples", Array.from(buffer.getChannelData(0)));
            } catch (e) {
              model.set("error", String(e && e.stack ? e.stack : e));
            }
            model.set("response_id", thisId);   // always bumped last, and always a fresh value ->
            model.save_changes();                // guaranteed change notification even on repeats/errors
          }
          model.on("change:request_id", doRender);
        }
        export default { render };
        """

    tone_bridge = ToneBridge()
    display(tone_bridge)   # must be displayed once so the JS side actually mounts and listens

    def wait_for_change(widget, trait):
        future = asyncio.Future()
        def getvalue(change):
            widget.unobserve(getvalue, trait)
            future.set_result(change.new)
        widget.observe(getvalue, trait)
        return future

    async def synth_tonejs(freq, cutoff, dur=0.5, timeout=15):
        """Drop-in async replacement for synth()'s oscillator+filter stage, rendered in-browser via Tone.js."""
        tone_bridge.duration = dur
        tone_bridge.params = [float(freq), float(cutoff)]
        this_id = tone_bridge.request_id + 1
        tone_bridge.request_id = this_id   # always a fresh int -> always fires change:request_id in JS
        try:
            while tone_bridge.response_id != this_id:
                await asyncio.wait_for(wait_for_change(tone_bridge, "response_id"), timeout=timeout)
        except asyncio.TimeoutError:
            raise TimeoutError(
                f"Tone.js render timed out after {timeout}s -- no response from the browser. "
                "Check the browser's JS console (F12) for the real error: likely Tone.js failed "
                "to load from the CDN, or the widget never mounted."
            )
        if tone_bridge.error:
            raise RuntimeError(f"Tone.js render failed in the browser:\\n{tone_bridge.error}")
        return np.array(tone_bridge.samples, dtype=np.float32)

    TONEJS_AVAILABLE = True
except ImportError:
    TONEJS_AVAILABLE = False
    print("anywidget not installed -- pip install anywidget to use SOUND_ENGINE='tonejs'")

## 1. The effects chain — the agent's "hands"

Two parameters, each discretized to 10 steps → a 100-action space. **Keep it small** so the
agent visibly learns inside a session. The agent doesn't synthesize anything — it picks a
**pitch-shift** amount and a **distortion drive** amount and applies them to the source sound
above, via [pedalboard](https://github.com/spotify/pedalboard) (Spotify's audio library) —
real, production-grade effects instead of hand-rolled DSP.

In [ ]:
SOUND_ENGINE = "python"   # "python" | "tonejs"  (tonejs needs Binder/local Jupyter -- see cell above)

from pedalboard import Pedalboard, PitchShift, Distortion

PITCH_SHIFTS = np.linspace(-12, 12, 10)   # semitones
DRIVE_DB     = np.linspace(0, 30, 10)     # distortion drive, dB (0 = clean)
N_PITCH, N_CUTOFF = len(PITCH_SHIFTS), len(DRIVE_DB)

def synth(pitch_idx, cutoff_idx, dur=None):
    board = Pedalboard([
        PitchShift(semitones=float(PITCH_SHIFTS[pitch_idx])),
        Distortion(drive_db=float(DRIVE_DB[cutoff_idx])),
    ])
    return normalize(board(SOURCE_AUDIO, SR).astype(np.float32))

async def synth_async(pitch_idx, cutoff_idx, dur=0.5):
    """synth(), or its Tone.js-rendered equivalent when SOUND_ENGINE == 'tonejs'."""
    if SOUND_ENGINE == "tonejs" and TONEJS_AVAILABLE:
        return await synth_tonejs(PITCH_SHIFTS[pitch_idx], DRIVE_DB[cutoff_idx], dur)
    return synth(pitch_idx, cutoff_idx, dur)

# preview one setting -- respects SOUND_ENGINE, unlike a plain synth() call
preview = await synth_async(7, 5)
display(Audio(preview, rate=SR))

## 2. The room — the agent's environment

By default a synthetic exponential-decay reverb so the notebook runs with no files.
**To play YOUR room:** record/Drop an impulse response (a short WAV — a clap or sine sweep
recorded in the space) and load it where marked. The agent will then be learning to play
*that* room.

In [ ]:
USE_MY_IR = False  # <-- set True after you add an impulse response file

# Kaggle: put room_ir.wav in /kaggle/working/ (file panel) or add it as a Dataset
# (then it lives at /kaggle/input/<your-dataset>/room_ir.wav -- edit the path below)
IR_PATH = "/kaggle/working/room_ir.wav"

if USE_MY_IR:
    from scipy.io import wavfile
    sr_ir, ir = wavfile.read(IR_PATH)
    ir = ir.astype(np.float32)
    if ir.ndim > 1: ir = ir.mean(axis=1)          # mono
    if sr_ir != SR:                                # resample to our SR
        ir = sps.resample(ir, int(len(ir)*SR/sr_ir))
    ir = ir / (np.max(np.abs(ir))+1e-9)
    IR = ir.astype(np.float32)
else:
    # frequency-dependent decay (highs damp faster, like a real room) + a few early reflections
    dur = 0.6
    t = np.linspace(0, dur, int(SR*dur))
    bands = [(20, 500, 3.0), (500, 3000, 5.0), (3000, SR/2*0.99, 9.0)]   # (lo, hi, decay)
    tail = np.zeros_like(t)
    for lo, hi, decay in bands:
        b, a = sps.butter(2, [lo/(SR/2), hi/(SR/2)], btype='band')
        tail += sps.lfilter(b, a, np.random.randn(len(t))) * np.exp(-decay*t)
    early = np.zeros_like(t)
    for tap_t, gain in [(0.005, 0.6), (0.015, 0.4), (0.03, 0.25)]:
        early[int(tap_t*SR)] = gain
    IR = (early + tail).astype(np.float32)
    IR = IR / (np.max(np.abs(IR)) + 1e-9)

def room(x):
    return sps.fftconvolve(x, IR)[:len(x)].astype(np.float32)

display(Audio(room(synth(7,5)), rate=SR))  # same note, now through the room

## 3. The goal sound — what the agent is chasing

By default the goal is **Alvin Lucier's *"I Am Sitting in a Room"* (1969)**, literalized: take
the source sound above and feed it through the room thousands of times — each pass is another
"playback and re-recording." The room's resonant frequencies get reinforced every iteration
while everything else decays away. The raw result of that process is unlistenable on its own
(it converges to a spiky click-train, not a clean tone — a real property of repeated
self-convolution, not a bug), so the dominant resonant frequencies it converges on are extracted
and resynthesized as a clean sustained tone — the actual finding, made audible. The agent never
gets a perfect match — it only gets one pass of its effects chain to approximate what 5000
iterations of physical re-recording revealed about the room. **The gap between the two is the
actual artifact.**

**To use your own goal instead:** set `USE_MY_GOAL = True` and point it at any WAV.

In [ ]:
USE_MY_GOAL = False  # <-- set True to chase your own sound instead of the Lucier process below

# Kaggle: put goal.wav in /kaggle/working/ or add it as a Dataset and edit the path
GOAL_PATH = "/kaggle/working/goal.wav"

GOAL_ITERATIONS = 5000   # "I Am Sitting in a Room": re-feed the source through the room this many times

if USE_MY_GOAL:
    from scipy.io import wavfile
    sr_g, g = wavfile.read(GOAL_PATH)
    g = g.astype(np.float32)
    if g.ndim > 1: g = g.mean(axis=1)
    if sr_g != SR: g = sps.resample(g, int(len(g)*SR/sr_g))
    goal_wet = normalize(g).astype(np.float32)
    GOAL_LABEL = "your sound"
else:
    lucier = SOURCE_AUDIO.copy()
    for _ in range(GOAL_ITERATIONS):
        lucier = normalize(room(lucier))
    # The raw iterated waveform converges to a persistently spiky/click-like signal (crest factor
    # ~8, confirmed structural across many IR/source designs -- not fixable by renormalizing or
    # compressing alone). What IS reliable: the iteration correctly finds which frequencies the
    # room favors. So extract those dominant resonant frequencies and resynthesize a clean,
    # sustained tone from them -- the actual Lucier mechanism, played back as something audible.
    freqs = np.fft.rfftfreq(len(lucier), 1/SR)
    mags = np.abs(np.fft.rfft(lucier))
    K = 6
    peak_idx = np.argsort(mags)[-K:]
    peak_freqs, peak_mags = freqs[peak_idx], mags[peak_idx]
    peak_mags = peak_mags / peak_mags.max()

    dur_goal = 3.0
    t_goal = np.linspace(0, dur_goal, int(SR*dur_goal), endpoint=False)
    rng_goal = np.random.default_rng(1)
    tone = np.zeros_like(t_goal)
    for f, m in zip(peak_freqs, peak_mags):
        tone += m * np.sin(2*np.pi*f*t_goal + rng_goal.uniform(0, 2*np.pi))
    fade = int(SR*0.1)
    env = np.ones_like(t_goal); env[:fade] = np.linspace(0,1,fade); env[-fade:] = np.linspace(1,0,fade)
    goal_wet = normalize((tone*env).astype(np.float32))
    GOAL_LABEL = f"the room's dominant resonances after {GOAL_ITERATIONS} trips through it (Lucier's \"I Am Sitting in a Room\"), resynthesized as a clean tone"

print("GOAL:", GOAL_LABEL)
display(Audio(goal_wet, rate=SR))

## 4. The reward — how "closeness" is measured

Three switchable modes. **This choice is the heart of the experiment** — "close" is not given,
it is *defined*. Switch modes and the agent converges on different sounds for the *same* goal.

- `spectrogram` — full time-frequency picture; **recommended default**. This is the mode that
  actually rewards *texture*: chorus, phaser, delay, modulation — anything that changes over
  time. Without it, those effects are invisible to the reward and the agent only ever learns to
  correct pitch, no matter how many effects you give it.
- `magspec` — time-averaged frequency balance (forgiving, musical, but throws away all timing
  information — chorus/phaser/delay become irrelevant to the score under this mode)
- `features` — a few summary numbers, e.g. brightness/energy (coarsest, smoothest to climb)

In [ ]:
REWARD_MODE = "spectrogram"   # "magspec" | "spectrogram" | "features"

from scipy.ndimage import zoom

def _magspec(x, nbins=64, smooth=5):
    f,t,S = sps.spectrogram(x, SR, nperseg=512)
    m = S.mean(axis=1)
    m = np.interp(np.linspace(0,len(m)-1,nbins), np.arange(len(m)), m)
    m = np.convolve(m, np.ones(smooth)/smooth, mode='same')   # smooth across bins: a clean,
                                                                # band-limited spectrum has narrow
                                                                # peaks that barely overlap unless
                                                                # frequencies match almost exactly,
                                                                # which makes the reward landscape
                                                                # a needle in a haystack; this keeps
                                                                # "magspec" actually forgiving
    m = np.log1p(m); return m/(np.linalg.norm(m)+1e-9)

def _spectrogram(x):
    f,t,S = sps.spectrogram(x, SR, nperseg=256)
    S = np.log1p(S); return S/(np.linalg.norm(S)+1e-9)

def _features(x):
    f,t,S = sps.spectrogram(x, SR, nperseg=512)
    m = S.mean(axis=1)+1e-9
    centroid = np.sum(f*m)/np.sum(m)        # brightness
    spread   = np.sqrt(np.sum(((f-centroid)**2)*m)/np.sum(m))
    energy   = np.log1p(np.sum(m))
    v = np.array([centroid/ (SR/2), spread/(SR/2), energy/20.0])
    return v/(np.linalg.norm(v)+1e-9)

def _resize_spec(S, shape):
    # rescale instead of truncating, so mismatched attempt/goal lengths don't bias the edges
    return zoom(S, (shape[0]/S.shape[0], shape[1]/S.shape[1]))

_REP_FNS = {"magspec":_magspec, "spectrogram":_spectrogram, "features":_features}
_GOAL_REP = _REP_FNS[REWARD_MODE](goal_wet)

def reward(agent_wet):
    rep = _REP_FNS[REWARD_MODE](agent_wet)
    if REWARD_MODE == "spectrogram" and rep.shape != _GOAL_REP.shape:
        rep = _resize_spec(rep, _GOAL_REP.shape)
    return -float(np.linalg.norm(rep - _GOAL_REP))

## 5. The agent — and **listening to it learn, step by step**

A simple value-learning agent (running-average bandit over the action grid) with decaying
exploration. `PLAY_EVERY` auto-plays the agent's current attempt as it goes, so you **hear the
search**: wild and wrong early, homing in later. Lower `STEP_PAUSE`/`PLAY_EVERY` for a faster
run, raise them to let the room sit with each attempt.

In [ ]:
EPISODES   = 120 if SOUND_ENGINE == "python" else 50   # tonejs has per-step browser round-trip latency -- keep it shorter
PLAY_EVERY = max(1, EPISODES // 8)      # play (and show) the agent's attempt every N steps
STEP_PAUSE = 0.0     # seconds to pause between steps (raise for live pacing)

action_values = np.zeros(N_PITCH*N_CUTOFF)   # incremental-mean value per action
visits = np.zeros(N_PITCH*N_CUTOFF)
rng = np.random.default_rng(1)

history, best_reward, best_action, best_wet = [], -1e9, None, None
attempts_audio = []   # collect for the "whole learning run" export

async def run_agent():
    global best_reward, best_action, best_wet
    for ep in range(EPISODES):
        eps = max(0.05, np.exp(-ep / 40))      # explore -> exploit (exponential decay)
        if rng.random() < eps or np.all(visits==0):
            a = int(rng.integers(N_PITCH*N_CUTOFF))
        else:
            masked = np.where(visits > 0, action_values, -np.inf)   # never exploit an unvisited action
            max_val = np.max(masked)
            candidates = np.where(masked == max_val)[0]
            a = int(rng.choice(candidates))                          # break ties randomly
        p, c = divmod(a, N_CUTOFF)
        dry = await synth_async(p, c); wet = normalize(room(dry))
        r = reward(wet)
        visits[a] += 1
        alpha = 1.0 / visits[a]
        action_values[a] += alpha * (r - action_values[a])           # incremental mean update
        r_smooth = r if not history else 0.9*history[-1] + 0.1*r      # smoothed for the plotted curve only
        history.append(r_smooth)
        attempts_audio.append(wet)
        if r > best_reward: best_reward, best_action, best_wet = r, (p,c), wet

        if ep % PLAY_EVERY == 0 or ep == EPISODES-1:
            print(f"step {ep:3d} | action (pitch_shift={PITCH_SHIFTS[p]:+.1f}st, drive={DRIVE_DB[c]:.0f}dB) | closeness {r:+.3f} | best {best_reward:+.3f}")
            display(Audio(wet, rate=SR, autoplay=False))
            if STEP_PAUSE: time.sleep(STEP_PAUSE)

    print(f"\nBest setting found: pitch_shift={PITCH_SHIFTS[best_action[0]]:+.1f}st, drive={DRIVE_DB[best_action[1]]:.0f}dB  (closeness {best_reward:+.3f})")

await run_agent()

## 6. Compare: goal vs. the agent's best — and the convergence curve

In [ ]:
print("GOAL:"); display(Audio(goal_wet, rate=SR))
print("AGENT'S BEST:"); display(Audio(best_wet, rate=SR))

fig, ax = plt.subplots(1,2, figsize=(12,3.2))
ax[0].plot(history, lw=1); ax[0].set_title("closeness over time (higher = closer)")
ax[0].set_xlabel("step"); ax[0].set_ylabel("reward")
# running best
rb = np.maximum.accumulate(history); ax[0].plot(rb, 'r--', lw=1, label="best so far"); ax[0].legend()

f,t,Sg = sps.spectrogram(goal_wet, SR, nperseg=256)
f,t,Sa = sps.spectrogram(best_wet, SR, nperseg=256)
ax[1].pcolormesh(np.log1p(Sg)); ax[1].set_title("goal spectrogram")
plt.tight_layout(); plt.show()

## 7. The whole learning run as one sound

Concatenate every attempt, in order, into a single clip — the entire wrong-to-right
trajectory as one evolving minute. Often the most striking artifact to keep, and it
sidesteps Colab's no-streaming limit entirely.

In [ ]:
clips = []
for a in attempts_audio:
    seg = normalize(a[:int(SR*0.25)].copy())   # .copy(): avoid mutating the stored attempt via the fade below
    fade = np.linspace(0, 1, 200)
    seg[:200] *= fade
    seg[-200:] *= fade[::-1]
    clips.append(seg)

run = np.concatenate(clips)  # 0.25s of each attempt, faded to avoid clicks at the seams
print(f"whole run: {len(attempts_audio)} attempts, {len(run)/SR:.1f}s")
display(Audio(run, rate=SR))

# save it
from scipy.io import wavfile
out_path = os.path.join(OUTPUT_DIR, "learning_run.wav")
wavfile.write(out_path, SR, (run/np.max(np.abs(run))*0.9*32767).astype(np.int16))
print("saved to", out_path)

## 8. Scaling up: from 2 knobs to a real effects rack

A 10×10 grid (100 actions) can be fully enumerated by a lookup-table bandit. A many-dimensional
continuous parameter vector cannot — even at just 2 values per parameter, 27 of them is 2^27
actions. So the agent itself has to change: instead of a table of per-action averages, it's now
a single **(1+1) evolution strategy** (a self-adapting hill-climber): keep the current best
parameter vector, nudge it with Gaussian noise, keep the nudge if it scored better, throw it
away if it didn't.

But the more important change is *what* it's controlling. Instead of 50 abstract numbers feeding
10 oscillators (where most of them barely affected the sound — phase, in particular, did almost
nothing to the reward), this is now a genuine **effects rack**: pitch shift, a Moog-style ladder
filter, chorus, phaser, distortion, delay, reverb, and a compressor — 27 parameters, but every
one of them is a real, distinct, audibly different knob, the way an actual modular synth or
pedalboard works. The question this section actually asks: **does more complexity get the agent
closer to the same Lucier-room target than the simple 2-knob version did, or does a bigger
search space just make the problem harder without buying anything?**

The exploration noise is scaled by `1/sqrt(N_PARAMS)` — the same trick AlphaZero uses when it
adds Dirichlet noise to its move probabilities "in inverse proportion to the approximate number
of legal moves": more dimensions/options means each one gets proportionally less noise, so the
total exploration pressure across the whole vector stays comparable as the space grows.

The step size (`sigma`) also self-adapts using the classic **1-in-5 success rule**: if more than
1 in 5 recent nudges are improving things, grow the steps (open ground, move faster); if fewer
than 1 in 5 are improving, shrink them (near a peak, be more careful).

In [ ]:
from pedalboard import Pedalboard, PitchShift, LadderFilter, Chorus, Phaser, Distortion, Delay, Reverb, Compressor

# (effect class, [(param_name, lo, hi, log_scale), ...]) -- a real rack, not abstract numbers
RACK_SPEC = [
    (PitchShift,   [("semitones", -12, 12, False)]),
    (LadderFilter, [("cutoff_hz", 80, 8000, True), ("resonance", 0.0, 0.9, False), ("drive", 1.0, 8.0, False)]),
    (Chorus,       [("rate_hz", 0.2, 4.0, False), ("depth", 0.0, 1.0, False), ("centre_delay_ms", 1.0, 20.0, False), ("feedback", 0.0, 0.5, False), ("mix", 0.0, 1.0, False)]),
    (Phaser,       [("rate_hz", 0.2, 4.0, False), ("depth", 0.0, 1.0, False), ("centre_frequency_hz", 200, 4000, True), ("feedback", 0.0, 0.5, False), ("mix", 0.0, 1.0, False)]),
    (Distortion,   [("drive_db", 0, 30, False)]),
    (Delay,        [("delay_seconds", 0.0, 0.4, False), ("feedback", 0.0, 0.6, False), ("mix", 0.0, 0.5, False)]),
    (Reverb,       [("room_size", 0.0, 1.0, False), ("damping", 0.0, 1.0, False), ("wet_level", 0.0, 0.6, False), ("dry_level", 0.2, 1.0, False), ("width", 0.0, 1.0, False)]),
    (Compressor,   [("threshold_db", -40, 0, False), ("ratio", 1.0, 10.0, False), ("attack_ms", 0.5, 50, False), ("release_ms", 10, 300, False)]),
]
N_PARAMS = sum(len(specs) for _, specs in RACK_SPEC)   # 27

def synth_rack(params, dur=None):
    """27 real effect parameters -> a Pedalboard chain applied to SOURCE_AUDIO."""
    params = np.clip(params, 0, 1)
    idx = 0
    plugins = []
    for cls, specs in RACK_SPEC:
        kwargs = {}
        for name, lo, hi, log_scale in specs:
            v = float(params[idx]); idx += 1
            val = lo * (hi/lo)**v if log_scale else lo + v*(hi-lo)
            kwargs[name] = float(val)
        plugins.append(cls(**kwargs))
    board = Pedalboard(plugins)
    return normalize(board(SOURCE_AUDIO, SR).astype(np.float32))

# preview a random rack setting
display(Audio(synth_rack(np.random.default_rng(2).uniform(0,1,N_PARAMS)), rate=SR))

## 9. Same goal, much bigger toolkit

Reuses `goal_wet` and the reward representation from sections 3-4 directly — **the exact same
Lucier-room target as the 2-knob version.** Nothing about "closeness" changes; only how many
ways the agent has to try to get there.

In [ ]:
# the exact same Lucier-room target as section 3 -- no separate goal, no separate reward needed
goal_wet50 = goal_wet
reward50 = reward

print("GOAL (rack):", GOAL_LABEL)
display(Audio(goal_wet50, rate=SR))

## 10. The agent grows up: from bandit to (1+1)-ES

No more `action_values` table — there's nothing to tabulate over an essentially infinite,
continuous action space. Instead: one running "current best" guess, perturbed and refined
step by step.

In [ ]:
EPISODES50   = 800
PLAY_EVERY50 = 80
STEP_PAUSE50 = 0.0

rng50 = np.random.default_rng(1)

BASE_SIGMA = 0.3                               # overall exploration "energy" budget, independent of dimensionality
sigma      = BASE_SIGMA / np.sqrt(N_PARAMS)    # per-parameter noise (AlphaZero scales its exploration
                                                # noise in inverse proportion to the action-space size, the same way)
SIGMA_FLOOR = sigma / 5     # tuned so the step size can't collapse so far it can never escape a rut
SIGMA_CEIL  = sigma * 10

best_params50 = rng50.uniform(0, 1, N_PARAMS)
best_wet50    = normalize(room(synth_rack(best_params50)))
best_reward50 = reward50(best_wet50)

history50, attempts_audio50 = [best_reward50], [best_wet50]
recent_successes = []   # rolling window for the classic 1-in-5 success rule (Rechenberg)

for ep in range(1, EPISODES50):
    candidate = np.clip(best_params50 + rng50.normal(0, sigma, N_PARAMS), 0, 1)
    wet = normalize(room(synth_rack(candidate)))
    r = reward50(wet)

    success = r > best_reward50
    recent_successes.append(success)
    if len(recent_successes) > 20:
        recent_successes.pop(0)
    if len(recent_successes) == 20:
        success_rate = np.mean(recent_successes)
        sigma = np.clip(sigma * (1.22 if success_rate > 0.2 else 1/1.22), SIGMA_FLOOR, SIGMA_CEIL)

    if success:
        best_params50, best_wet50, best_reward50 = candidate, wet, r

    history50.append(0.9*history50[-1] + 0.1*r)   # smoothed for the plot, as in section 5
    attempts_audio50.append(wet)

    if ep % PLAY_EVERY50 == 0 or ep == EPISODES50-1:
        print(f"step {ep:3d} | sigma {sigma:.4f} | closeness {r:+.3f} | best {best_reward50:+.3f}")
        display(Audio(wet, rate=SR, autoplay=False))
        if STEP_PAUSE50: time.sleep(STEP_PAUSE50)

print(f"\nBest closeness found: {best_reward50:+.3f}")

## 11. Compare, and the whole rack-search run as one sound — does more complexity actually help?

In [ ]:
print("GOAL (rack):"); display(Audio(goal_wet50, rate=SR))
print("AGENT'S BEST (rack):"); display(Audio(best_wet50, rate=SR))
print(f"For comparison, the 2-knob version's best was {best_reward:+.3f}; the rack's best is {best_reward50:+.3f}")

fig, ax = plt.subplots(1,2, figsize=(12,3.2))
ax[0].plot(history50, lw=1); ax[0].set_title("closeness over time (rack search)")
ax[0].set_xlabel("step"); ax[0].set_ylabel("reward")
rb50 = np.maximum.accumulate(history50); ax[0].plot(rb50, 'r--', lw=1, label="best so far"); ax[0].legend()

f,t,Sg50 = sps.spectrogram(goal_wet50, SR, nperseg=256)
ax[1].pcolormesh(np.log1p(Sg50)); ax[1].set_title("goal spectrogram")
plt.tight_layout(); plt.show()

In [ ]:
from scipy.io import wavfile

clips50 = []
for a in attempts_audio50:
    seg = normalize(a[:int(SR*0.25)].copy())
    fade = np.linspace(0, 1, 200)
    seg[:200] *= fade
    seg[-200:] *= fade[::-1]
    clips50.append(seg)

run50 = np.concatenate(clips50)
print(f"whole run (rack search): {len(attempts_audio50)} attempts, {len(run50)/SR:.1f}s")
display(Audio(run50, rate=SR))

out_path50 = os.path.join(OUTPUT_DIR, "learning_run_rack.wav")
wavfile.write(out_path50, SR, (run50/np.max(np.abs(run50))*0.9*32767).astype(np.int16))
print("saved to", out_path50)

---
### Where to take it next
- **Swap `REWARD_MODE`** (cell 4) and re-run — same goal, different idea of "close," different result. Good live moment.
- **Upload your room IR** (cell 2) — the agent learns to play *your* space.
- **Upload your own goal sound** (cell 3) — chase a voice or field recording; listen to the near-miss.
- **CLAP reward (advanced):** replace `reward()` with cosine similarity between CLAP embeddings of `agent_wet` and the goal (or a *text* goal). Same architecture, semantic instead of acoustic "closeness" — connects this to the embedding-space experiment.
